In [ ]:
#use environment.yml
import pandas as pd
import numpy as np
import http.client
import requests
import json
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os
import holidays
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor, AdaBoostRegressor
)
import joblib
import mlflow
import mlflow.sklearn
from pathlib import Path
from xgboost import XGBRegressor
import lightgbm as lgb
from scipy.sparse import csr_matrix


Prendre top 20stations
Travailler sur le mois de Mars/avril (car après pour projet Lead, données sauvergardés à partir de mars/avril) -> comparer les résulats du modèle

In [ ]:
#Charger données PIerre
PATH = "../01_Data/"
dataset = pd.read_parquet(f"{PATH}dataset_export.parquet")

TOP20_STATION_LIST = PATH +'top20_station_list.csv'
top20_station= pd.read_csv(TOP20_STATION_LIST)

#mois à inclure
month_studied = ("2025-03", "2025-04")

dataset.describe().T
dataset.info()
dataset.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19674960 entries, 0 to 19674959
Data columns (total 23 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   station_id                 object        
 1   date                       datetime64[ns]
 2   date_hour                  datetime64[ns]
 3   year                       int32         
 4   month                      int32         
 5   day                        int32         
 6   jour_semaine               object        
 7   hour                       int64         
 8   num_bikes_taken            int64         
 9   num_bikes_dropped          int64         
 10  net_flow                   int64         
 11  temp                       float32       
 12  relative_humidity          float32       
 13  precipitation_total        float32       
 14  average_wind_speed         float32       
 15  coco                       int64         
 16  is_holiday                 bool   

,station_id,date,date_hour,year,month,day,jour_semaine,hour,num_bikes_taken,num_bikes_dropped,...,precipitation_total,average_wind_speed,coco,is_holiday,coco_label,coco_group,precipitation_total_round,precipitation_total_bin,temp_round,temp_bin
0,2261.04,2024-11-01,2024-11-01 00:00:00,2024,11,1,Vendredi,0,0,0,...,0.0,7.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
1,2261.04,2024-11-01,2024-11-01 01:00:00,2024,11,1,Vendredi,1,0,0,...,0.0,15.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",24.0,"(22.3, 24.75]"
2,2261.04,2024-11-01,2024-11-01 02:00:00,2024,11,1,Vendredi,2,0,0,...,0.0,7.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
3,2261.04,2024-11-01,2024-11-01 03:00:00,2024,11,1,Vendredi,3,0,0,...,0.0,22.700001,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
4,2261.04,2024-11-01,2024-11-01 04:00:00,2024,11,1,Vendredi,4,0,0,...,0.0,11.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",22.0,"(19.85, 22.3]"


### Features engineering
Prédiction Random forest avec données temporelles -> apprentissage supervisé
Random Forest supoose que les observations sont indépendantes
-> création de lags: Utiliser les valeurs passées pour prédire la valeur future.
-> Fonctionnalités temporelles : Extraire des informations de la date (Jour de la semaine, mois, année, jour de l'année, trimestre, vacances).
-> Statistiques glissantes (Rolling statistics) : Calculer des moyennes ou des écarts-types sur une fenêtre de temps glissante (ex: moyenne des 7 derniers jours)



In [ ]:
def prepare_features(dataset, month_studied= None):
    dataset = dataset.copy()
    
    dataset["date"] = pd.to_datetime(dataset["date"], format="%Y-%m-%d", errors="raise")
    dataset = dataset.sort_values(["station_id","date_hour"])
            
    # --- Filtrer les dates si nécessaire ---
    if month_studied is not None:           
        start = pd.to_datetime(month_studied[0] + "-01")
        end   = pd.to_datetime(month_studied[1] + "-01") + pd.offsets.MonthEnd(1)

        dataset = dataset[
            (dataset["date"] >= start) &
            (dataset["date"] <= end)
        ]
    # Target (t+1h) + time-series features
    dataset["y"] = dataset.groupby("station_id")["net_flow"].shift(-1)
    
    #Features temporelles supplémentaires
    dataset['hour_sin'] = np.sin(2*np.pi*dataset['hour']/24)
    dataset['hour_cos'] = np.cos(2*np.pi*dataset['hour']/24)
    dataset['day_sin'] = np.sin(2*np.pi*dataset['day']/31)
    dataset['day_cos'] = np.cos(2*np.pi*dataset['day']/31)
    dataset['month_sin'] = np.sin(2*np.pi*dataset['month']/12)
    dataset['month_cos'] = np.cos(2*np.pi*dataset['month']/12)
    dataset['is_weekend'] = dataset['jour_semaine'].isin(['Samedi','Dimanche']).astype(int)
    dataset['is_peak'] = (((dataset['hour']>=6) & (dataset['hour']<10)) | ((dataset['hour']>=16) & (dataset['hour']<20))).astype(int)
    
    
    # --- Features météo ---
    dataset['cold_weather'] = (dataset['temp']<5).astype(int)
    dataset['hot_weather'] = (dataset['temp']>30).astype(int)
    dataset['heavy_rain'] = (dataset['precipitation_total']>5).astype(int)
    
    # --- Lag features ---
    
    lags = [1,2,24]
    for l in lags:
        dataset[f"net_flow_lag_{l}"] = dataset.groupby("station_id")["net_flow"].shift(l)

    dataset["net_flow_roll_3"] = dataset.groupby("station_id")["net_flow"].transform(
        lambda x: x.shift(1).rolling(3).mean()
    )

    dataset["net_flow_roll_24"] = dataset.groupby("station_id")["net_flow"].transform(
        lambda x: x.shift(1).rolling(24).mean()
    )
        
    # optional lags for pickups/drops (if present)
    for col in ["num_bikes_taken", "num_bikes_dropped"]:
        if col in dataset.columns:
            dataset[f"{col}_lag_1"] = dataset.groupby("station_id")[col].shift(1)

    # --- Features finales pour le modèle ---
    features = [
        #'hour_sin','hour_cos','day_sin','day_cos','month_sin','month_cos',
        #'is_weekend','is_peak',
        "station_id", 'year', 'month', 'day', 'hour',
        'temp','precipitation_total','relative_humidity','average_wind_speed',
        #'cold_weather','hot_weather','heavy_rain',
        'num_bikes_taken_lag_1','num_bikes_dropped_lag_1',
        'net_flow_lag_1','net_flow_lag_2','net_flow_lag_24','net_flow_roll_3','net_flow_roll_24',
        'jour_semaine', 'coco_group', 'is_holiday', 'coco'
        
    ]
    
    # Drop lignes avec NaN issues des lags
    needed = ["y"] + [f"net_flow_lag_{l}" for l in lags] + ["net_flow_roll_3","net_flow_roll_24"]
    dataset = dataset.dropna(subset=needed).reset_index(drop=True)
    
    # Cible
    target = 'net_flow'
    
    cols_to_keep = [col for col in features if col in dataset.columns]
    if 'net_flow' not in cols_to_keep:
        cols_to_keep.append('net_flow')
        
    filtered_df = dataset[cols_to_keep].copy()
    
    return filtered_df, features, target


dataset_all, features, target = prepare_features(dataset)#, month_studied=month_studied)


Nombre de lignes : 8735


In [ ]:
FILE = "data/dataset_station_preprocessed.parquet"
if FILE in os.listdir():
        os.remove(FILE)
dataset_all.to_parquet(FILE, index=False, compression="snappy")

PATH="../data"
if FILE in os.listdir():
        os.remove(FILE)
dataset_all.to_parquet(FILE, index=False, compression="snappy")

dataset_fe = dataset_all[dataset_all["station_id"].isin(top20_station)].copy()

TOP20_FILE = "data/dataset_top20_stations.parquet"
if TOP20_FILE in os.listdir():
        os.remove(TOP20_FILE)
dataset_fe.to_parquet(TOP20_FILE, index=False, compression="snappy")

print("Nombre de lignes :", len(dataset_fe))

Split train/test pour données temporelles:
- pas de validation croisée aléatoire (K-Fold classique). Il faut respecter la structure temporelle. Le jeu d'entrainement doit précéder le jeu de test

In [ ]:
# Load Data
features = [
        "station_id", 'year', 'month', 'day', 'hour',
        'temp','precipitation_total','relative_humidity','average_wind_speed',
        'num_bikes_taken_lag_1','num_bikes_dropped_lag_1',
        'net_flow_lag_1','net_flow_lag_2','net_flow_lag_24','net_flow_roll_3','net_flow_roll_24',
        'jour_semaine', 'coco_group', 'is_holiday', 'coco'  
    ]
# Cible
target = 'net_flow'

DATA_DIR = '/data/'

FILE_NAME = 'historical_data.parquet'
dataset_fe = pd.read_parquet(DATA_DIR + FILE_NAME, columns=features + [target])

# Downcast numeric types
for col in dataset_fe.select_dtypes(include=['float64']).columns:
    dataset_fe[col] = dataset_fe[col].astype('float32')

for col in dataset_fe.select_dtypes(include=['int64']).columns:
    dataset_fe[col] = dataset_fe[col].astype('int32')


In [ ]:
#Split chronologique
cut = int(len(dataset_fe) * 0.8)
train_df = dataset_fe.iloc[:cut].copy()
test_df  = dataset_fe.iloc[cut:].copy()

#TimeSeriesSplit de scikit-learn pour éviter les fuites temporelles
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

y_pred_naive_test = test_df["net_flow"].to_numpy()
rmse_naive = float(np.sqrt(mean_squared_error(y_test, y_pred_naive_test)))
mae_naive  = float(mean_absolute_error(y_test, y_pred_naive_test))
r2_naive   = float(r2_score(y_test, y_pred_naive_test))

print("=== Naive baseline on TEST (y(t+1)=net_flow(t)) ===")
print(f"RMSE: {rmse_naive:.4f} | MAE: {mae_naive:.4f} | R2: {r2_naive:.4f}\n")
    
 # Preprocessing: numeric + categorical
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype) == "category"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

print("Numeric features:", num_cols)
print("Categorical features:", cat_cols)

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, num_cols),
        ("cat", categorical_preprocess, cat_cols),
    ],
    remainder="drop"
)

# Models
models = [
    ("LinearRegression", LinearRegression()),
    ("Ridge(alpha=1.0)", Ridge(alpha=1.0, random_state=42)),
    ("Lasso(alpha=0.001)", Lasso(alpha=0.001, random_state=42, max_iter=20000)),
    ("RandomForest(300,depth=14)", RandomForestRegressor(
        n_estimators=300, max_depth=14, random_state=42, n_jobs=-1
    )),
    ("ExtraTrees(500,depth=16)", ExtraTreesRegressor(
        n_estimators=500, max_depth=16, random_state=42, n_jobs=-1
    )),
        ("XGBoost", XGBRegressor(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.0, reg_lambda=1.0,
        random_state=42, n_jobs=-1
    )),
]

def score(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return rmse, mae, r2
    
# Train + score all models
results = []

X_train_prep = preprocess.fit_transform(X_train)
X_test_prep  = preprocess.transform(X_test)

X_train_prep = csr_matrix(X_train_prep)
X_test_prep  = csr_matrix(X_test_prep)

for name, model in models:
    
    model.fit(X_train_prep, y_train)
    pred = model.predict(X_test_prep)

    rmse, mae, r2 = score(y_test, pred)
    results.append({
        "model": name,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "rmse_gain_vs_naive": rmse_naive - rmse
    })

res = pd.DataFrame(results).sort_values("rmse")

print("=== Benchmark on TEST (lower RMSE is better) ===")
print(res.to_string(index=False))

print("\n=== Models beating naive (rmse < naive) ===")
better = res[res["rmse"] < rmse_naive][["model","rmse","mae","r2","rmse_gain_vs_naive"]]
print(better.to_string(index=False))

print("\nNaive TEST reference:")
print(f"RMSE: {rmse_naive:.4f} | MAE: {mae_naive:.4f} | R2: {r2_naive:.4f}")

Train shape: (128487, 20), Test shape: (32122, 20)
=== Naive baseline on TEST (y(t+1)=net_flow(t)) ===
RMSE: 0.0000 | MAE: 0.0000 | R2: 1.0000

Numeric features: ['year', 'month', 'day', 'hour', 'temp', 'precipitation_total', 'relative_humidity', 'average_wind_speed', 'num_bikes_taken_lag_1', 'num_bikes_dropped_lag_1', 'net_flow_lag_1', 'net_flow_lag_2', 'net_flow_lag_24', 'net_flow_roll_3', 'net_flow_roll_24', 'is_holiday', 'coco']
Categorical features: ['station_id', 'jour_semaine', 'coco_group']
=== Benchmark on TEST (lower RMSE is better) ===
  model     rmse      mae       r2  rmse_gain_vs_naive
XGBoost 7.671762 4.371929 0.384471           -7.671762

=== Models beating naive (rmse < naive) ===
Empty DataFrame
Columns: [model, rmse, mae, r2, rmse_gain_vs_naive]
Index: []

Naive TEST reference:
RMSE: 0.0000 | MAE: 0.0000 | R2: 1.0000


In [ ]:
# Hyperparameter tuning for XGBoost only

xgb_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

param_dist = {
    "model__n_estimators": [400, 600, 800, 1000, 1200],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__reg_alpha": [0.0, 0.1, 1.0],
    "model__reg_lambda": [0.5, 1.0, 2.0, 5.0]
}

tscv = TimeSeriesSplit(n_splits=4)

search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

best_xgb = search.best_estimator_

pred_best = best_xgb.predict(X_test)
rmse_best, mae_best, r2_best = score(y_test, pred_best)

print("=== Best XGB (tuned) on TEST ===")
print(f"RMSE: {rmse_best:.4f} | MAE: {mae_best:.4f} | R2: {r2_best:.4f}")
print(f"Naive RMSE: {rmse_naive:.4f} | Gain: {rmse_naive - rmse_best:.4f}")

Fitting 4 folds for each of 1 candidates, totalling 4 fits


/opt/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=30. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best params: {'model__subsample': 1.0, 'model__reg_lambda': 5.0, 'model__reg_alpha': 1.0, 'model__n_estimators': 800, 'model__min_child_weight': 10, 'model__max_depth': 6, 'model__learning_rate': 0.01, 'model__colsample_bytree': 1.0}
Best CV RMSE: 6.5262686014175415
=== Best XGB (tuned) on TEST ===
RMSE: 7.6575 | MAE: 4.3159 | R2: 0.3868
Naive RMSE: 0.0000 | Gain: -7.6575


Commentaire final : Le benchmark de plusieurs algorithmes montre que XGBoost obtient la meilleure performance sur le jeu de test avec un RMSE de 1.923, soit une amélioration d’environ 21 % par rapport à la baseline naïve (RMSE = 2.434).

Une recherche d’hyperparamètres par RandomizedSearchCV a été réalisée. Le modèle initial conserve la meilleure performance sur le jeu de test final. Ce dernier est donc retenu comme modèle final.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

final_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=search.best_params_['model__n_estimators'],
        learning_rate=search.best_params_['model__learning_rate'],
        max_depth=search.best_params_['model__max_depth'],
        subsample=search.best_params_['model__subsample'],
        colsample_bytree=search.best_params_['model__colsample_bytree'],
        reg_alpha=search.best_params_['model__reg_alpha'],
        min_child_weight=search.best_params_['model__min_child_weight'],
        reg_lambda=search.best_params_['model__reg_lambda'],
        random_state=42,
        n_jobs=-1
    ))
])


#Fit the pipeline
final_pipe.fit(X_train, y_train)

Path("model").mkdir(exist_ok=True)

joblib.dump(final_pipe, "model/citibike_forecast_model.joblib")

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Prédictions

In [ ]:
model = XGBRegressor()
model.fit(X_train, y_train)

# Log model to MLflow
with mlflow.start_run():
    mlflow.xgboost.log_model(model, artifact_path="citibike_xgb_model")

In [22]:
def predict_from_user_date(dataset, request_datetime, features, model_path, weather_df, update_columns,
                           station_id=None):

    df = dataset.copy()
    request_datetime = pd.to_datetime(request_datetime)

    # charger modèle
    model = joblib.load(model_path)

    # extraire caractéristiques temporelles
    month_req = request_datetime.month
    dow_req = request_datetime.dayofweek

    # filtrer historique
    df["date_hour"] = pd.to_datetime(df["date_hour"])

    df_filtered = df[
        (df["date_hour"].dt.month == month_req) &
        (df["date_hour"].dt.dayofweek == dow_req)
    ]

    # filtrer station si nécessaire
    if station_id is not None:
        df_filtered = df_filtered[df_filtered["station_id"] == station_id]

    # trier
    df_filtered = df_filtered.sort_values(["station_id", "date_hour"])

    # prendre la dernière observation
    X = df_filtered[features].iloc[-1:]

    # Get US holidays
    us_holidays = holidays.US(years=request_datetime.year)
    
    # Données météo actuelles
    for col in update_columns:
        if col in X.columns and col in weather_df.columns:
            X[col] = weather_df[col].values[0]
        if col == "is_holiday":
            X[col] = 1 if request_datetime.date() in us_holidays else 0

    # prédiction
    prediction = model.predict(X)[0]

    return prediction

In [42]:
request_hours = "20:00" #from today
station = '6072.06'

In [ ]:
# Get current weather data for NYC station
conn = http.client.HTTPSConnection("meteostat.p.rapidapi.com")

headers = {
    'x-rapidapi-key': os.getenv("RAPIDAPI_KEY"),
    'x-rapidapi-host': "meteostat.p.rapidapi.com"
}

NYC_station_id = "KNYC0"
tz = "Europe/Berlin"
url = f"/stations/hourly?station={NYC_station_id}&start={datetime.today().date()}&end={datetime.today().date()}&tz={tz}"

conn.request("GET", url, headers=headers)
res = conn.getresponse()

weather_json = json.loads(res.read().decode("utf-8"))

weather_data = weather_json["data"][-1]

weather_df = pd.DataFrame(weather_data, index=[0])
print('weather data loaded into dataframe')

# Convert time
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Rename columns
weather_df = weather_df.rename(columns={
    "rhum": "relative_humidity",
    "prcp": "precipitation_total",
    "wspd": "average_wind_speed",
})
print('renaming columns done')

weather_df['coco_group'] = weather_df['coco'].map({
    1: 'Pas de pluie', 2: 'Pas de pluie', 3: 'Pas de pluie',
    4: 'Pas de pluie', 5: 'Pas de pluie', 6: 'Pas de pluie',
    8: 'Pluie/Neige', 9: 'Pluie/Neige', 10: 'Pluie/Neige',
    11: 'Pluie/Neige', 12: 'Pluie/Neige', 13: 'Pluie/Neige',
    14: 'Pluie/Neige', 15: 'Pluie/Neige', 16: 'Pluie/Neige',
    18: 'Pluie/Neige', 19: 'Pluie/Neige', 20: 'Pluie/Neige',
    21: 'Pluie/Neige', 22: 'Pluie/Neige', 23: 'Pluie/Neige',
    24: 'Pluie/Neige', 25: 'Pluie/Neige', 26: 'Pluie/Neige',
    27: 'Pluie/Neige',
    17: 'Averse de pluie'
})

#remove useless columns
keep_columns = ['temp', 'relative_humidity', 'precipitation_total', 'average_wind_speed', 'coco', 'coco_group'] 
weather_df = weather_df[keep_columns]
print('keeping only specified columns')

weather data loaded into dataframe
renaming columns done
keeping only specified columns


In [43]:
hour, minute = map(int, request_hours.split(":"))
request_datetime = datetime.today().replace(hour=hour, minute=minute, second=0, microsecond=0)

update_columns = ['temp', 'relative_humidity', 'precipitation_total', 'average_wind_speed', 'coco', 'coco_group', 'is_holiday'] 

pred = predict_from_user_date(
        dataset=dataset_fe,
        request_datetime=request_datetime,
        features=features,
        model_path="model/citibike_xgb_pipeline.joblib",
        station_id=station,
        weather_df=weather_df,
        update_columns=update_columns
    )
print(f"Station {station} - Prévision net_flow prochaine heure on {request_datetime} : {pred:.1f}")

Station 6072.06 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : 2.6


In [16]:
results = []

hour, minute = map(int, request_hours.split(":"))
request_datetime = datetime.today().replace(hour=hour, minute=minute, second=0, microsecond=0)

for station in dataset_fe['station_id'].unique():

    pred = predict_from_user_date(
        dataset=dataset_fe,
        request_datetime=request_datetime,
        features=features,
        model_path="model/citibike_xgb_pipeline.joblib",
        station_id=station,
        weather_df=weather_df,
        keep_columns=keep_columns
    )

    results.append({
        "station_id": station,
        "prediction": pred
    })
    print(f"Station {station} - Prévision net_flow prochaine heure on {request_datetime} : {pred:.1f}")
forecast_df = pd.DataFrame(results)

Station 4455.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : 0.7
Station 5137.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : -0.2
Station 5247.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : 1.2
Station 5343.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : 0.8
Station 5359.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : 1.1
Station 5379.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : -0.3
Station 5506.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : -0.1
Station 5669.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : -0.4
Station 5779.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : -0.2
Station 5872.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : -0.3
Station 5980.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : -0.3
Station 5997.1 - Prévision net_flow prochaine heure on 2026-03-14 20:00:00 : 1.5
Station 6364.1 - Prév